# 13 · Bloomberg Terminal Data

Connects to a local Bloomberg Professional Terminal session via `blpapi`.
Fetches TTF forward curves M1–M24, historical prices/OI/volume, and a
multi-asset panel (TTF, Brent, JKM, NBP, EUA carbon).

**Requirements:**
- Bloomberg Terminal must be running and logged in
- `pip install blpapi` (Bloomberg Python SDK)

**Graceful fallback:** every cell checks `BBG_OK` and prints a clear
message if Bloomberg is not available, so the notebook runs without errors.

In [ ]:
import sys, os, warnings
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
from datetime import date
warnings.filterwarnings('ignore')

for _c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (_c / 'src' / 'agsi_client').exists():
        sys.path.insert(0, str(_c)); os.chdir(_c)
        print(f'\u2705 Root: {_c}'); break

from src.agsi_client.bloomberg_client import (
    BloombergClient, BloombergNotAvailableError,
    TTF_MONTHLY_TICKERS, TTF_DA_TICKER, MULTI_ASSET_TICKERS,
)

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
N_MONTHS   = 24          # TTF curve tenors to fetch
HIST_START = '2020-01-01'
BBG_HOST   = 'localhost'
BBG_PORT   = 8194
# ─────────────────────────────────────────────────────────────────────────────

BBG_OK  = False
client  = None
BBG_ERR = ''

try:
    client = BloombergClient(host=BBG_HOST, port=BBG_PORT)
    BBG_OK = True
    print('\u2705 Bloomberg Terminal connected')
except BloombergNotAvailableError as e:
    BBG_ERR = str(e)
    print(f'\u26a0\ufe0f  Bloomberg not available:\n{e}')
    print('\nThis notebook will run in demo mode — all data cells are skipped.')
except Exception as e:
    BBG_ERR = str(e)
    print(f'\u26a0\ufe0f  Unexpected error: {e}')

# Paths
RAW_DIR  = Path('data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)
TTF_CSV_DATABENTO = RAW_DIR / 'ttf_curve.csv'
TTF_CSV_BBG       = RAW_DIR / 'ttf_curve_bbg.csv'

print(f'Status  : {"CONNECTED" if BBG_OK else "NOT AVAILABLE"}')
print(f'Raw dir : {RAW_DIR.resolve()}')

## 1 · TTF Forward Curve M1–M24 + Day-Ahead

Fetch the latest snapshot from Bloomberg and compare it to the Databento
`ttf_curve.csv` if that file exists.

In [ ]:
if not BBG_OK:
    print('\u26a0\ufe0f  Bloomberg not available — skipping TTF curve fetch.')
else:
    # ── Bloomberg snapshot ──────────────────────────────────────────────────
    bbg_curve = client.get_ttf_curve(n_months=N_MONTHS)
    print(f'Bloomberg curve snapshot ({date.today()}):')
    display(bbg_curve)

    # ── Compare to Databento CSV ─────────────────────────────────────────────
    if TTF_CSV_DATABENTO.exists():
        db_curve = pd.read_csv(TTF_CSV_DATABENTO, index_col=0, parse_dates=True)
        db_curve.index = pd.to_datetime(db_curve.index).tz_localize(None)
        db_latest = db_curve.iloc[-1]
        db_date   = db_curve.index[-1].date()
        print(f'\nDatabento latest row: {db_date}')
    else:
        db_latest = None
        print('\n\u2139\ufe0f  ttf_curve.csv (Databento) not found — comparison skipped.')

    # ── Chart: Bloomberg curve (+ Databento overlay if available) ───────────
    month_cols = [c for c in bbg_curve.columns if c.startswith('M') and c[1:].isdigit()]
    month_cols = sorted(month_cols, key=lambda c: int(c[1:]))
    x_vals     = [int(c[1:]) for c in month_cols]
    y_bbg      = [float(bbg_curve.iloc[0].get(c, np.nan)) for c in month_cols]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x_vals, y=y_bbg, mode='lines+markers', name='Bloomberg',
        line=dict(color='#6366f1', width=2.5), marker=dict(size=6),
    ))

    if db_latest is not None:
        db_m_cols = [c for c in db_latest.index if c.startswith('M') and c[1:].isdigit()]
        db_m_cols = sorted(db_m_cols, key=lambda c: int(c[1:]))
        db_x      = [int(c[1:]) for c in db_m_cols]
        db_y      = [float(db_latest.get(c, np.nan)) for c in db_m_cols]
        fig.add_trace(go.Scatter(
            x=db_x, y=db_y, mode='lines+markers', name=f'Databento ({db_date})',
            line=dict(color='#f97316', width=1.5, dash='dot'), marker=dict(size=5),
        ))

    # Day-ahead annotation
    da_price = float(bbg_curve.iloc[0].get('DA', np.nan))
    if not np.isnan(da_price):
        fig.add_hline(
            y=da_price, line_color='#22c55e', line_dash='dash',
            annotation_text=f'DA: \u20ac{da_price:.2f}',
            annotation_font_size=10,
        )

    fig.update_layout(
        template='plotly_white', height=460,
        title=f'TTF Forward Curve M1\u2013M{N_MONTHS} — Bloomberg vs Databento',
        xaxis=dict(title='Tenor', tickvals=x_vals, ticktext=[f'M{x}' for x in x_vals]),
        yaxis_title='\u20ac/MWh',
        legend=dict(orientation='h', y=-0.15),
    )
    fig.show()

    # Diff table
    if db_latest is not None:
        common = [c for c in month_cols if c in db_latest.index]
        if common:
            diff = pd.DataFrame({
                'Bloomberg': [float(bbg_curve.iloc[0].get(c, np.nan)) for c in common],
                'Databento': [float(db_latest.get(c, np.nan)) for c in common],
            }, index=common)
            diff['\u0394 (BBG\u2212DB)'] = diff['Bloomberg'] - diff['Databento']
            print('\nPrice comparison (\u20ac/MWh):')
            display(diff.round(3))

## 2 · Historical M1 Price, Volume & Open Interest

BDH request for `TGE1 Comdty` since `HIST_START`: price (PX_LAST), open
interest (OPEN_INT) and volume (PX_VOLUME). The three series are plotted on
a dual-axis chart.

In [ ]:
if not BBG_OK:
    print('\u26a0\ufe0f  Bloomberg not available — skipping historical fetch.')
else:
    m1_hist = client.get_historical_multi(
        'TGE1 Comdty',
        fields=['PX_LAST', 'OPEN_INT', 'PX_VOLUME'],
        start=HIST_START,
    )
    print(f'TGE1 Comdty: {len(m1_hist)} rows  ({m1_hist.index[0].date()} \u2192 {m1_hist.index[-1].date()})')
    display(m1_hist.tail())

    if not m1_hist.empty and 'PX_LAST' in m1_hist.columns:
        fig = make_subplots(
            rows=3, cols=1, shared_xaxes=True,
            subplot_titles=[
                'TTF M1 Settlement Price (\u20ac/MWh)',
                'Open Interest (contracts)',
                'Volume (contracts/day)',
            ],
            vertical_spacing=0.06, row_heights=[0.50, 0.25, 0.25],
        )

        fig.add_trace(go.Scatter(
            x=m1_hist.index, y=m1_hist['PX_LAST'],
            name='M1 Price', line=dict(color='#6366f1', width=1.5),
        ), row=1, col=1)

        if 'OPEN_INT' in m1_hist.columns:
            fig.add_trace(go.Scatter(
                x=m1_hist.index, y=m1_hist['OPEN_INT'],
                name='Open Interest', fill='tozeroy',
                line=dict(color='#3b82f6', width=1),
                fillcolor='rgba(59,130,246,0.15)',
            ), row=2, col=1)

        if 'PX_VOLUME' in m1_hist.columns:
            fig.add_trace(go.Bar(
                x=m1_hist.index, y=m1_hist['PX_VOLUME'],
                name='Volume', marker_color='#94a3b8', opacity=0.75,
            ), row=3, col=1)

        fig.update_yaxes(title_text='\u20ac/MWh', row=1, col=1)
        fig.update_yaxes(title_text='OI', row=2, col=1)
        fig.update_yaxes(title_text='Volume', row=3, col=1)
        fig.update_layout(
            height=660, template='plotly_white',
            title='TTF M1 (TGE1 Comdty) — Price, OI & Volume from Bloomberg',
            showlegend=False,
        )
        fig.show()

        # Summary stats
        price = m1_hist['PX_LAST'].dropna()
        print(f'\nPrice summary ({HIST_START} \u2192 today):')
        print(f'  Min   : \u20ac{price.min():.2f}/MWh')
        print(f'  Max   : \u20ac{price.max():.2f}/MWh')
        print(f'  Mean  : \u20ac{price.mean():.2f}/MWh')
        print(f'  Latest: \u20ac{price.iloc[-1]:.2f}/MWh')

## 3 · Replace / Augment `ttf_curve.csv` with Bloomberg Data

If Bloomberg is available, fetch the full historical curve M1–M24 and write
it to `data/raw/ttf_curve.csv`, replacing the Databento source.
The original Databento file is kept as `ttf_curve_databento_backup.csv`.

In [ ]:
if not BBG_OK:
    print('\u26a0\ufe0f  Bloomberg not available — ttf_curve.csv unchanged.')
else:
    import shutil

    # Backup existing Databento CSV if present
    if TTF_CSV_DATABENTO.exists():
        backup = RAW_DIR / 'ttf_curve_databento_backup.csv'
        shutil.copy(TTF_CSV_DATABENTO, backup)
        print(f'\u2705 Backed up Databento CSV \u2192 {backup.name}')

    print(f'\nFetching full historical curve (M1\u2013M{N_MONTHS}) from Bloomberg...')
    print('This may take a few minutes for 24 tenors.')

    hist_curve = client.get_ttf_historical_curve(
        n_months=N_MONTHS,
        start=HIST_START,
    )

    if hist_curve.empty:
        print('\u26a0\ufe0f  No historical curve data returned.')
    else:
        hist_curve.to_csv(TTF_CSV_DATABENTO)   # replaces existing file
        hist_curve.to_csv(TTF_CSV_BBG)         # also save to bbg-specific path
        print(f'\u2705 Bloomberg curve saved \u2192 {TTF_CSV_DATABENTO}  ({len(hist_curve)} rows)')
        print(f'\u2705 Bloomberg curve saved \u2192 {TTF_CSV_BBG}')
        display(hist_curve.tail())

        # Preview chart
        m_cols = sorted(
            [c for c in hist_curve.columns if c.startswith('M') and c[1:].isdigit()],
            key=lambda c: int(c[1:]),
        )
        fig = go.Figure()
        for col in m_cols[:6]:  # first 6 tenors
            fig.add_trace(go.Scatter(
                x=hist_curve.index, y=hist_curve[col],
                name=col, line=dict(width=1.2),
            ))
        fig.update_layout(
            template='plotly_white', height=420,
            title='TTF Bloomberg Historical Curve — M1\u2013M6 Settlement Prices',
            yaxis_title='\u20ac/MWh',
            legend=dict(orientation='h', y=-0.15),
        )
        fig.show()

## 4 · Multi-Asset Panel: TTF vs Brent, JKM, NBP, EUA Carbon

Fetches settlement prices for five energy benchmarks from Bloomberg:

| Ticker | Market |
|---|---|
| `TGE1 Comdty` | TTF Natural Gas M1 (€/MWh) |
| `CO1 Comdty` | ICE Brent Crude M1 ($/bbl) |
| `JKMM1 Comdty` | JKM LNG M1 ($/mmBtu) |
| `NBPG1 Comdty` | NBP UK Natural Gas M1 (p/therm) |
| `CFI2 Comdty` | EUA Carbon (EU ETS) (€/t CO₂) |

In [ ]:
if not BBG_OK:
    print('\u26a0\ufe0f  Bloomberg not available — skipping multi-asset fetch.')
else:
    series_dict = {}
    for label, ticker in MULTI_ASSET_TICKERS.items():
        try:
            hist = client.get_historical(ticker, 'PX_LAST', start=HIST_START)
            if not hist.empty:
                series_dict[label] = hist['PX_LAST']
                print(f'  \u2705 {label:<18} ({ticker}): {len(hist)} rows')
            else:
                print(f'  \u26a0\ufe0f  {label:<18} ({ticker}): no data')
        except Exception as e:
            print(f'  \u274c {label:<18} ({ticker}): {e}')

    if series_dict:
        multi_df = pd.DataFrame(series_dict).sort_index()

        # ── Normalised price chart (rebased to 100) ──────────────────────────
        rebased = multi_df.div(multi_df.iloc[0]) * 100

        palette = ['#6366f1', '#f97316', '#22c55e', '#3b82f6', '#94a3b8']
        fig = go.Figure()
        for i, col in enumerate(rebased.columns):
            fig.add_trace(go.Scatter(
                x=rebased.index, y=rebased[col],
                name=col, line=dict(color=palette[i % len(palette)], width=1.5),
            ))
        fig.add_hline(y=100, line_color='black', line_width=0.8, line_dash='dot')
        fig.update_layout(
            template='plotly_white', height=460,
            title=f'Multi-Asset Energy — Rebased to 100 ({HIST_START})',
            yaxis_title='Rebased Price (100 = start)',
            legend=dict(orientation='h', y=-0.15),
        )
        fig.show()

        # ── Correlation heatmap ───────────────────────────────────────────────
        ret_df  = np.log(multi_df / multi_df.shift(1)).dropna()
        corr    = ret_df.corr()
        n       = len(corr)

        fig2 = go.Figure(go.Heatmap(
            z=corr.values, x=corr.columns, y=corr.index,
            colorscale='RdBu', zmid=0, zmin=-1, zmax=1,
            text=[[f'{v:.2f}' for v in row] for row in corr.values],
            texttemplate='%{text}', textfont_size=11,
        ))
        fig2.update_layout(
            template='plotly_white', height=420,
            title='Daily Log-Return Correlation — Multi-Asset Energy',
            xaxis=dict(tickangle=20),
        )
        fig2.show()

        # ── Rolling 60-day correlation vs TTF M1 ─────────────────────────────
        ttf_ret = ret_df.get('TTF M1')
        if ttf_ret is not None:
            fig3 = go.Figure()
            for col in ret_df.columns:
                if col == 'TTF M1':
                    continue
                roll_corr = ttf_ret.rolling(60).corr(ret_df[col])
                fig3.add_trace(go.Scatter(
                    x=roll_corr.index, y=roll_corr,
                    name=col, line=dict(width=1.3),
                ))
            fig3.add_hline(y=0, line_color='black', line_width=0.8)
            fig3.update_layout(
                template='plotly_white', height=400,
                title='60-Day Rolling Correlation vs TTF M1',
                yaxis_title='Pearson r',
                legend=dict(orientation='h', y=-0.15),
            )
            fig3.show()

        # ── Current snapshot table ────────────────────────────────────────────
        latest_prices = client.get_field(
            list(MULTI_ASSET_TICKERS.values()),
            ['PX_LAST'],
        )
        latest_prices.index = pd.Index(MULTI_ASSET_TICKERS.keys(), name='Asset')
        print('\nCurrent settlement prices from Bloomberg:')
        display(latest_prices.rename(columns={'PX_LAST': 'Price'}).round(3))

## 5 · Export Bloomberg Curve to `data/raw/ttf_curve_bbg.csv`

Saves the Bloomberg TTF curve (M1–M24 historical) to a dedicated file so
both data sources can coexist. Also prints a coverage summary.

In [ ]:
if not BBG_OK:
    print('\u26a0\ufe0f  Bloomberg not available — no file written.')
    if TTF_CSV_BBG.exists():
        existing = pd.read_csv(TTF_CSV_BBG, index_col=0, parse_dates=True)
        print(f'\u2139\ufe0f  Existing {TTF_CSV_BBG.name}: {len(existing)} rows')
        display(existing.tail(3))
else:
    # hist_curve may already exist from cell 3; re-fetch if needed
    try:
        curve_to_export = hist_curve
    except NameError:
        print('Fetching Bloomberg historical curve...')
        curve_to_export = client.get_ttf_historical_curve(
            n_months=N_MONTHS, start=HIST_START,
        )

    if curve_to_export.empty:
        print('\u26a0\ufe0f  No data to export.')
    else:
        curve_to_export.to_csv(TTF_CSV_BBG)
        print(f'\u2705 Saved \u2192 {TTF_CSV_BBG}  ({len(curve_to_export)} rows)')

        # Coverage summary
        m_cols = sorted(
            [c for c in curve_to_export.columns if c.startswith('M') and c[1:].isdigit()],
            key=lambda c: int(c[1:]),
        )
        coverage = pd.DataFrame({
            'Tenor':  m_cols,
            'Count':  [curve_to_export[c].count() for c in m_cols],
            'First':  [curve_to_export[c].dropna().index[0].date() if curve_to_export[c].count() > 0 else None for c in m_cols],
            'Last':   [curve_to_export[c].dropna().index[-1].date() if curve_to_export[c].count() > 0 else None for c in m_cols],
            'LatestPrice': [curve_to_export[c].dropna().iloc[-1] if curve_to_export[c].count() > 0 else np.nan for c in m_cols],
        })
        print('\nCoverage summary:')
        display(coverage.round(3))

        # Close Bloomberg session cleanly
        client.close()
        print('\n\u2705 Bloomberg session closed.')